<a href="https://colab.research.google.com/github/Vish204/RAG/blob/main/Document_search_github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install PyMuPDF
!pip install sentence-transformers
!pip install faiss-cpu  # or faiss-gpu if you have GPU support
!pip install ipywidgets

In [ ]:
import os
import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
from google.colab import files

# Upload multiple files
uploaded_files = files.upload()

# Function to extract text from a single PDF file and split it into paragraphs
def extract_text_from_pdf(pdf_path):
    """Extracts text from a single PDF file and splits it into paragraphs."""
    text = ""
    with fitz.open(pdf_path) as pdf:
        for page in pdf:
            text += page.get_text() + "\n"
    return text.strip().split('\n\n')  # Split into paragraphs


# Extract text from all uploaded PDFs
documents = []
for pdf_file in uploaded_files.keys():
    paragraphs = extract_text_from_pdf(pdf_file)
    documents.append(paragraphs)

print(f" Loaded {len(documents)} PDFs successfully!")

# Load the pre-trained Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model Loaded Successfully!")

# Generate embeddings for all paragraphs
doc_embeddings = []
paragraph_counts = []

for paragraphs in documents:
    embeddings = model.encode(paragraphs)
    doc_embeddings.append(embeddings)
    paragraph_counts.append(len(paragraphs))

doc_embeddings = np.vstack(doc_embeddings)
print("Embedding Shape:", doc_embeddings.shape)

# Build FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

print(f" FAISS Index Built! Total Paragraphs Indexed: {index.ntotal}")


def search_top_k_with_pdf(query, top_k=3):
    """Search relevant paragraphs for a query."""
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, top_k)

    results = []

    for i, idx in enumerate(indices[0]):
        if idx >= 0:
            cumulative_count = 0
            for doc_index, count in enumerate(paragraph_counts):
                if idx < cumulative_count + count:
                    paragraph_index = idx - cumulative_count
                    paragraph = documents[doc_index][paragraph_index]
                    break
                cumulative_count += count

            results.append(
                f"\nDocument {doc_index + 1}, Paragraph {paragraph_index + 1}:\n{paragraph}\n"
            )

    return results


# ----------- SIMPLE QUERY LOOP (replaces widgets) -------------

while True:
    query = input("\nEnter your query (or type 'exit'): ")

    if query.lower() == "exit":
        print("Search ended.")
        break

    results = search_top_k_with_pdf(query, top_k=3)

    print("\n Search Results:")
    print("-" * 60)

    for i, result in enumerate(results, 1):
        print(f"\nTop {i} Match:")
        print(result)
        print("-" * 60)